# ECHO-ET Hourly Evapotranspiration — Ogallala ROI Pipeline
URL-list driven, chunked download → dask-threaded ROI filtering → CF-1.8 NetCDF.
No logins. Resumable. Corrupt-stall watchdog included.

**Data:** ECHO-ET hourly ET (ECOSTRESS-based, random forest on ERA5-hourly +
MODIS VI + ECOSTRESS 70-m ET, day/night EC-corrected, 194 flux sites), 0.05°,
global, hourly, 2018-2022, float32 GeoTIFF, **units W m-2**, fills -32767
(water/nodata/>1000) and -32768 (year-round snow/ice). Gridded validation vs
ECOSTRESS ET: r2 = 0.95, RMSE = 29 W m-2, PBIAS = 0.08%.
File hours are **local solar time (NOT UTC)** — this notebook processes the
`Local_solar_time/` tree and labels the time axis accordingly.

**Fair use (please read):** inform J. Xiao (j.xiao@unh.edu) / J. Bu
(jingyi.bu@unh.edu) before any manuscript; co-authorship may apply.
Citation: Bu, J., Xiao, J., Fisher, J.B., Luo, Y. (2026), Nature
Communications, in press. Product page: https://globalecology.unh.edu/data/ECHO-ET.html

**Zero-corruption design:** byte-verified resumable downloads (size + TIFF
magic + HTML-trap + rasterio open) with atomic rename; single-granule
redownloads; pure deterministic dask filter tasks (no shared state, results
re-sorted before serial appends); duplicate-timestep guard; fsync'd logs;
MD5-verified Drive copies; hourly backups. See Cell 4 for the full list.

In [ ]:
!pip -q install rasterio geopandas netCDF4 beautifulsoup4 tqdm lxml dask

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE  = True    # True -> process only TEST_LIMIT files end-to-end (dry run)
TEST_LIMIT = 100     # files processed when TEST_MODE is True

BATCH_SIZE           = 10000          # files per download/process cycle
MIN_FREE_BYTES       = 40 * 2**30    # pause downloads when scratch free space is below this
SPACE_CHECK_INTERVAL = 30 * 60       # re-check free space at least this often (seconds)
BACKUP_INTERVAL      = 60 * 60       # .nc (+ logs) -> Drive this often (seconds)
MAX_RETRIES          = 5             # per-file download/verify attempts
SYNC_EVERY           = 100           # ds.sync() every N appended timesteps
COMPRESS_LEVEL       = 1             # zlib level for the CF .nc
CHUNK                = 32 * 1024 * 1024
MAX_DEPTH            = 4             # crawl depth below BASE_URL (year/month needs 2)
SIMPLIFY_DEG         = 0.01
CORRUPT_STALL_MINUTES = 20           # stop + warn after this long with zero successes

N_DOWNLOAD_WORKERS = 8               # polite concurrent downloads (UNH academic server)
PARALLEL_FILES     = 64              # max files in flight per sub-batch
DASK_WORKERS       = 8               # threads for the filtering stage
DASK_BATCH         = 32              # files per dask.compute() call (memory bound)
TARGET_RES_DEG     = 0.01            # fallback common grid (non lat/lon inputs only)
TIME_UNITS         = "seconds since 2000-01-01 00:00:00 UTC"
TIME_CALENDAR      = "proleptic_gregorian"
# =======================================================================


In [ ]:
import os, re, json, time, shutil, gc, glob, threading
from datetime import datetime, timezone
from urllib.parse import urljoin, urlsplit, quote, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import requests
import dask
from dask import delayed
from tqdm.auto import tqdm
import geopandas as gpd
import rasterio
import rasterio.warp
from rasterio.windows import from_bounds
import netCDF4 as nc4
import logging

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("echo_et")
dask.config.set(scheduler="threads", num_workers=DASK_WORKERS)

# ---------- paths (Drive = durable, scratch = fast) ----------
BASE_URL   = "https://data.globalecology.unh.edu/data/ECHO-ET/Local_solar_time/"
DRIVE_OUT  = "/content/drive/MyDrive/ECHOET_UNH"
TMP_DIR    = "/content/echoet_raw"
NC_LOCAL   = "/content/echoet_nc/ECHOET_Ogallala.nc"
LOCAL_NC_DIR = "/content/echo_nc"
URLS_TXT   = "/content/urls.txt"
TIF_TXT    = "/content/tif_urls.txt"
LIST_CACHE = f"{DRIVE_OUT}/_filelist_ECHOET.json"
PROCESSED_TXT = f"{DRIVE_OUT}/processed_files.txt"
STATE_JSON    = f"{DRIVE_OUT}/backup_state.json"
CORRUPT_LOG   = f"{DRIVE_OUT}/corrupted_files.txt"
FAILED_LOG    = f"{DRIVE_OUT}/failed_files.txt"
UNPARSED_LOG  = f"{DRIVE_OUT}/unparsed_dates.txt"
SKIPPED_LOG   = f"{DRIVE_OUT}/skipped_files.txt"
MASK_NPZ      = f"{DRIVE_OUT}/roi_mask_cache.npz"
NC_DRIVE_LATEST = f"{DRIVE_OUT}/ECHOET_ROI_latest.nc"
NC_DRIVE_FINAL  = f"{DRIVE_OUT}/ECHOET_ROI_FINAL.nc"

# ---------- readme constants ----------
FILL             = np.float32(-9999.0)   # .nc missing-value flag
SNOW_FILL        = np.float32(-32768.0)  # readme: year-round snow/ice lands
WATER_FILL       = np.float32(-32767.0)  # readme: water bodies (+ nodata, values >1000)
MAX_VALID_VALUE  = np.float32(1000.0)
ET_UNITS         = "W m-2"
ET_STANDARD_NAME = "surface_upward_latent_heat_flux"
ET_VALID_RANGE   = np.array([0.0, 1000.0], dtype=np.float32)
PLANE_CONST_MIN_PX = 200
GIBBERISH_CONST_FRAC = 0.995
CITATION = ("Bu, J., Xiao, J., Fisher, J.B., Luo, Y. (2026) NASA's ECOSTRESS "
            "satellite reveals widespread midday depression in ecosystem "
            "evapotranspiration. Nature Communications, in press.")
PRODUCT_PAGE = "https://globalecology.unh.edu/data/ECHO-ET.html"
TU, TC = "hours since 2000-01-01 00:00:00", "proleptic_gregorian"

for _d in (TMP_DIR, os.path.dirname(NC_LOCAL), DRIVE_OUT):
    os.makedirs(_d, exist_ok=True)

# ---------- HTTP: one tuned pool, kept hot by concurrent workers ----------
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 ColabDownloader/1.0"})
try:
    from urllib3.util import Retry
except ImportError:
    from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

def _tune_session(s, pool=32):
    retry = Retry(total=5, backoff_factor=1.0,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("HEAD", "GET"), raise_on_status=False)
    ad = HTTPAdapter(pool_connections=16, pool_maxsize=pool, max_retries=retry)
    s.mount("https://", ad)
    s.mount("http://", ad)
    return s

_tune_session(SESSION)
_LOG_LOCK = threading.Lock()
_tls = threading.local()

def get_session():
    s = getattr(_tls, "s", None)
    if s is None:
        s = requests.Session()
        s.headers.update(SESSION.headers)
        _tune_session(s)
        _tls.s = s
    return s

def safe_url(u):
    p = urlsplit(u); q = f"?{p.query}" if p.query else ""
    return f"{p.scheme}://{p.netloc}{quote(unquote(p.path), safe='/')}{q}"

def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

def disk_free():
    return shutil.disk_usage("/content").free

# ---------- corruption stall watchdog (>20 min of pure failure -> stop) ----------
class CorruptStall(RuntimeError):
    # Raised when downloads/filters fail continuously with zero successes.
    pass

class StallWatchdog:
    def __init__(self, limit_min):
        self.lock = threading.Lock()
        self.limit = limit_min * 60.0
        self.stall_start = None
        self.n_fail = 0
    def report(self, ok):
        # Thread-safe. Returns False once failures run dry of successes too long.
        with self.lock:
            now = time.time()
            if ok:
                self.stall_start = None
                self.n_fail = 0
                return True
            if self.stall_start is None:
                self.stall_start = now
            self.n_fail += 1
            return (now - self.stall_start) <= self.limit
    def summary(self):
        with self.lock:
            secs = (time.time() - self.stall_start) if self.stall_start else 0.0
            return self.n_fail, secs

WATCHDOG = StallWatchdog(CORRUPT_STALL_MINUTES)

def note_outcome(ok, what):
    if not WATCHDOG.report(ok):
        n, secs = WATCHDOG.summary()
        raise CorruptStall(
            "No successful %s for %.1f minutes (%d consecutive failures). "
            "Stopping to avoid burning bandwidth on a dead source - just re-run to resume."
            % (what, secs / 60.0, n))

# ---------- server crawl (h5ai JSON + HTML fallback, parallel) ----------
CRAWL_WORKERS, CRAWL_RETRIES = 16, 3

def h5ai_json(url):
    r = get_session().get(safe_url(url) + "?json", timeout=60,
                          headers={"X-Requested-With": "XMLHttpRequest",
                                   "Accept": "application/json"})
    if r.status_code != 200:
        raise IOError(f"HTTP {r.status_code}")
    if r.text.lstrip()[:1] not in ("{", "["):
        raise ValueError("non-JSON body (h5ai API disabled?)")
    return r.json()

def html_listing(url):
    r = get_session().get(safe_url(url), timeout=60); r.raise_for_status()
    from bs4 import BeautifulSoup
    try:    soup = BeautifulSoup(r.text, "lxml")
    except Exception: soup = BeautifulSoup(r.text, "html.parser")
    items = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith(("#", "javascript", "mailto:")): continue
        absu = safe_url(urljoin(url, href))
        if not absu.startswith(BASE_URL): continue
        items.append({"href": absu, "is_dir": href.endswith("/"), "size": None})
    return items

def fetch_listing(url):
    last = None
    for attempt in range(1, CRAWL_RETRIES + 1):
        server_ok = False
        try:
            items = h5ai_json(url); server_ok = True
            if items: return url, items
        except Exception as e: last = e
        try:
            items = html_listing(url); server_ok = True
            if items: return url, items
        except Exception as e: last = e
        if server_ok: return url, []
        time.sleep(min(2 ** attempt, 8))
    tqdm.write(f"  ! listing failed after {CRAWL_RETRIES} tries: {url} ({last})")
    return url, []

def crawl(url, max_depth=MAX_DEPTH):
    root = url if url.endswith("/") else url + "/"
    try: get_session().get(safe_url(root), timeout=60)
    except Exception: pass
    visited, queue, files = {root}, [(root, 0)], []
    lock = threading.Lock()
    pbar = tqdm(total=1, desc="Crawl (dirs)", unit="dir")
    with ThreadPoolExecutor(max_workers=CRAWL_WORKERS) as ex:
        while queue:
            futures = {ex.submit(fetch_listing, d): (d, dep) for d, dep in queue}
            nxt = []
            for fut in as_completed(futures):
                u, items = fut.result()
                d, dep = futures[fut]
                pbar.update(1)
                for it in items:
                    name = unquote(it["href"].rstrip("/").rsplit("/", 1)[-1])
                    if it["is_dir"]:
                        if name.startswith("."): continue
                        sub = it["href"] if it["href"].endswith("/") else it["href"] + "/"
                        if sub not in visited and dep + 1 <= max_depth:
                            visited.add(sub); nxt.append((sub, dep + 1))
                    elif not name.lower().endswith((".html", ".htm", ".md5")):
                        with lock: files.append({"url": it["href"], "name": name, "size": it["size"]})
            pbar.total += len(nxt); pbar.refresh()
            queue = nxt
    pbar.close()
    return files

def head_size(url):
    try:
        h = get_session().head(safe_url(url), allow_redirects=True, timeout=60)
        if "Content-Length" in h.headers: return int(h.headers["Content-Length"])
    except Exception: pass
    return None

def fill_sizes(files):
    todo = [f for f in files if f["size"] is None]
    if not todo: return
    log(f"Fetching sizes for {len(todo)} files ({CRAWL_WORKERS} parallel)...")
    with ThreadPoolExecutor(max_workers=CRAWL_WORKERS) as ex:
        futs = {ex.submit(head_size, f["url"]): f for f in todo}
        for fut in tqdm(as_completed(futs), total=len(futs), desc="HEAD sizes", unit="file"):
            futs[fut]["size"] = fut.result()

# ---------- download (resumable, atomic, optionally quiet) + verify ----------
def _write_list_if_changed(path, urls):
    """Write a URL list only if missing or changed; never truncate to empty."""
    if not urls:
        raise RuntimeError("Refusing to write empty URL list to %s" % path)
    text = "\n".join(urls)
    if os.path.exists(path):
        with open(path) as _f:
            if _f.read() == text:
                return False
    with open(path, "w") as _f:
        _f.write(text)
    return True

def download(url, dest, expected_size=None, quiet=False, session=None):
    url, tmp = safe_url(url), dest + ".part"
    pos = os.path.getsize(tmp) if os.path.exists(tmp) else 0
    headers = {"Range": f"bytes={pos}-"} if pos else {}
    S = session or SESSION
    with S.get(url, headers=headers, stream=True, timeout=(30, 180)) as r:
        if r.status_code == 416:
            os.replace(tmp, dest); return
        r.raise_for_status()
        resume = (r.status_code == 206 and pos > 0)
        if not resume: pos = 0
        total = int(r.headers.get("Content-Length", 0)) + (pos if resume else 0) or expected_size
        bar = None
        if not quiet:
            bar = tqdm(total=total, initial=pos, unit="B", unit_scale=True, unit_divisor=1024,
                       desc=os.path.basename(dest)[:40], leave=False)
        try:
            with open(tmp, "ab" if resume else "wb") as fh:
                for chunk in r.iter_content(chunk_size=CHUNK):
                    if chunk:
                        fh.write(chunk)
                        if bar is not None: bar.update(len(chunk))
        finally:
            if bar is not None: bar.close()
    if expected_size is not None and os.path.getsize(tmp) != expected_size:
        raise IOError(f"size mismatch: got {os.path.getsize(tmp)}, expected {expected_size}")
    os.replace(tmp, dest)

def verify_tif(path, expected_size=None):
    if expected_size is not None and os.path.getsize(path) != expected_size:
        return False, "size mismatch"
    try:
        with open(path, "rb") as fh: head = fh.read(512)
        if head[:2] not in (b"II", b"MM"): return False, "bad TIFF magic bytes"
        if b"<html" in head.lower(): return False, "HTML error page, not data"
    except Exception as e: return False, str(e)
    try:
        with rasterio.open(path) as src:
            _ = src.bounds
            src.read(1, out_shape=(1, min(src.height, 64) or 1, min(src.width, 64) or 1))
    except Exception as e: return False, f"rasterio read failed: {e}"
    return True, "ok"

def _append_line(path, text):
    with _LOG_LOCK:
        with open(path, "a") as f:
            f.write(text + "\n")

def log_corrupt(entry):
    _append_line(CORRUPT_LOG, json.dumps(entry))
def log_failed(url):
    _append_line(FAILED_LOG, url)
def log_skipped(url, why):
    _append_line(SKIPPED_LOG, json.dumps({"url": url, "reason": why}))

DT_RE_HOUR = re.compile(r"(\d{8})(\d{2})\.tif$", re.I)
DT_RE_DATE = re.compile(r"(\d{4})(\d{2})(\d{2})")

def parse_dt(name):
    m = DT_RE_HOUR.search(name)
    if m:
        try: return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H")
        except ValueError: pass
    m = DT_RE_DATE.search(name)
    if m:
        try: return datetime(*map(int, m.groups()))
        except ValueError: pass
    return None

def timestep_key(dt):
    return dt.strftime("%Y%m%dT%H%M")


In [ ]:
# ---------- ROI polygon ----------
_gdf = gpd.read_file(SHP_PATH)
if _gdf.crs is None or _gdf.crs.to_epsg() != 4326: _gdf = _gdf.to_crs(epsg=4326)
try:    ROI = _gdf.union_all().simplify(SIMPLIFY_DEG, preserve_topology=True)
except AttributeError: ROI = _gdf.unary_union.simplify(SIMPLIFY_DEG, preserve_topology=True)
log(f"ROI loaded, bounds (lon/lat): {[round(v, 3) for v in ROI.bounds]}")

try:
    from shapely import contains_xy
    def poly_mask(geom, x, y): return contains_xy(geom, x, y)
except ImportError:
    from shapely.vectorized import contains as _vc
    def poly_mask(geom, x, y): return _vc(geom, x, y)

GRID, ds, nc_dirty = None, None, False
next_backup = time.time() + BACKUP_INTERVAL
_proc_buffer = []
PROC_LOG_FLUSH_EVERY = 50

def init_grid(sample_path):
    global GRID
    with rasterio.open(sample_path) as src:
        ref_w, ref_h, ref_crs, ref_t = src.width, src.height, src.crs, src.transform
        roi_r = gpd.GeoSeries([ROI], crs=4326).to_crs(src.crs).iloc[0]
        minx, miny, maxx, maxy = roi_r.bounds
        win = from_bounds(minx, miny, maxx, maxy, transform=ref_t)
        c0 = max(int(np.floor(win.col_off)), 0); r0 = max(int(np.floor(win.row_off)), 0)
        c1 = min(int(np.ceil(win.col_off + win.width)), ref_w)
        r1 = min(int(np.ceil(win.row_off + win.height)), ref_h)
        win = rasterio.windows.Window(c0, r0, c1 - c0, r1 - r0)
        wt  = src.window_transform(win)
        nx, ny = int(win.width), int(win.height)
        xs = wt.c + (np.arange(nx) + 0.5) * wt.a
        ys = wt.f + (np.arange(ny) + 0.5) * wt.e
        X2, Y2 = np.meshgrid(xs, ys)
        lon_f, lat_f = rasterio.warp.transform(src.crs, "EPSG:4326", X2.ravel(), Y2.ravel())
        lon2 = np.asarray(lon_f).reshape(ny, nx)
        lat2 = np.asarray(lat_f).reshape(ny, nx)
        mask = poly_mask(ROI, lon2, lat2)
        GRID = dict(win=win, wt=wt, ny=ny, nx=nx, ref_w=ref_w, ref_h=ref_h,
                    ref_crs=ref_crs, xs=xs, ys=ys,
                    lat2=lat2.astype(np.float32), lon2=lon2.astype(np.float32), mask=mask)
        GRID["idx"] = np.flatnonzero(np.asarray(mask, dtype=bool))
    log(f"Grid ready: subset {ny}x{nx} px, {int(mask.sum())} px inside ROI, CRS={ref_crs}")

def save_mask_cache():
    np.savez(MASK_NPZ,
             ref_w=np.int64(GRID["ref_w"]), ref_h=np.int64(GRID["ref_h"]),
             crs_wkt=np.array(GRID["ref_crs"].to_wkt()),
             win=np.array([GRID["win"].col_off, GRID["win"].row_off,
                           GRID["win"].width, GRID["win"].height], dtype=np.float64),
             wt=np.array([GRID["wt"].a, GRID["wt"].b, GRID["wt"].c,
                          GRID["wt"].d, GRID["wt"].e, GRID["wt"].f], dtype=np.float64),
             mask=np.asarray(GRID["mask"], dtype=np.uint8),
             lat2=GRID["lat2"], lon2=GRID["lon2"], xs=GRID["xs"], ys=GRID["ys"],
             roi_bounds=np.array(ROI.bounds, dtype=np.float64))
    log(f"Mask cache saved: {MASK_NPZ}")

def load_mask_cache():
    global GRID
    if not os.path.exists(MASK_NPZ): return False
    try:
        z = np.load(MASK_NPZ, allow_pickle=False)
        if not np.allclose(z["roi_bounds"], np.array(ROI.bounds), atol=1e-9):
            log("Mask cache ROI mismatch - rebuilding."); return False
        from rasterio.windows import Window as _Win
        try: from rasterio.transform import Affine as _Af
        except ImportError: from affine import Affine as _Af
        GRID = dict(win=_Win(*[float(v) for v in z["win"]]),
                    wt=_Af(*[float(v) for v in z["wt"]]),
                    ny=int(z["mask"].shape[0]), nx=int(z["mask"].shape[1]),
                    ref_w=int(z["ref_w"]), ref_h=int(z["ref_h"]),
                    ref_crs=rasterio.crs.CRS.from_wkt(str(z["crs_wkt"])),
                    xs=np.asarray(z["xs"]), ys=np.asarray(z["ys"]),
                    lat2=np.asarray(z["lat2"]).astype(np.float32),
                    lon2=np.asarray(z["lon2"]).astype(np.float32),
                    mask=np.asarray(z["mask"]).astype(bool))
        if os.path.exists(NC_LOCAL):
            with nc4.Dataset(NC_LOCAL, "r") as _r:
                if len(_r.dimensions["y"]) != GRID["ny"] or len(_r.dimensions["x"]) != GRID["nx"]:
                    log("Mask cache grid != resumed .nc grid - rebuilding.")
                    GRID = None; return False
        GRID["idx"] = np.flatnonzero(GRID["mask"])
        log(f"Mask cache reused: {GRID['ny']}x{GRID['nx']} px, "
            f"{int(GRID['mask'].sum())} inside ROI (no rebuild)")
        return True
    except Exception as e:
        log(f"Mask cache unreadable ({e}) - rebuilding."); GRID = None; return False

def ensure_grid(raw_list):
    global GRID
    if GRID is not None: return True
    if load_mask_cache(): return True
    for e in raw_list:
        if os.path.exists(e["local"]):
            init_grid(e["local"]); save_mask_cache(); return True
    return False

def _grid_fingerprint():
    import hashlib
    h = hashlib.md5()
    h.update(str(GRID["wt"]).encode()); h.update(GRID["ref_crs"].to_wkt().encode())
    h.update(("%d/%d" % (GRID["ny"], GRID["nx"])).encode())
    return h.hexdigest()

ZLIB = COMPRESS_LEVEL > 0
def _clevel(): return max(COMPRESS_LEVEL, 1)

def create_nc(path):
    ny, nx = GRID["ny"], GRID["nx"]; wt = GRID["wt"]
    d = nc4.Dataset(path, "w", format="NETCDF4")
    d.createDimension("time", None); d.createDimension("y", ny); d.createDimension("x", nx)
    t = d.createVariable("time", "f8", ("time",), fill_value=False)
    t.units, t.calendar, t.standard_name, t.axis = TU, TC, "time", "T"
    t.long_name = "local solar time (hour beginning; NOT UTC)"
    geo = GRID["ref_crs"].is_geographic
    yv = d.createVariable("y", "f8", ("y",), fill_value=False)
    yv.units = "degrees_north" if geo else "m"
    yv.standard_name = "latitude" if geo else "projection_y_coordinate"; yv.axis = "Y"; yv[:] = GRID["ys"]
    xv = d.createVariable("x", "f8", ("x",), fill_value=False)
    xv.units = "degrees_east" if geo else "m"
    xv.standard_name = "longitude" if geo else "projection_x_coordinate"; xv.axis = "X"; xv[:] = GRID["xs"]
    lat = d.createVariable("lat", "f4", ("y", "x"), zlib=ZLIB, complevel=_clevel(), fill_value=False)
    lat.standard_name, lat.units = "latitude", "degrees_north"; lat[:] = GRID["lat2"]
    lon = d.createVariable("lon", "f4", ("y", "x"), zlib=ZLIB, complevel=_clevel(), fill_value=False)
    lon.standard_name, lon.units = "longitude", "degrees_east"; lon[:] = GRID["lon2"]
    m = d.createVariable("roi_mask", "i1", ("y", "x"), zlib=ZLIB, complevel=_clevel(), fill_value=False)
    m.long_name = "1 inside ROI polygon, 0 outside"; m[:] = GRID["mask"].astype(np.int8)
    crs = d.createVariable("crs", "i4")
    wkt = GRID["ref_crs"].to_wkt()
    crs.crs_wkt, crs.spatial_ref = wkt, wkt
    crs.GeoTransform = f"{wt.c} {wt.a} {wt.b} {wt.f} {wt.d} {wt.e}"
    if GRID["ref_crs"].to_epsg(): crs.epsg_code = GRID["ref_crs"].to_epsg()
    et = d.createVariable("ET", "f4", ("time", "y", "x"), fill_value=FILL,
                          zlib=ZLIB, complevel=_clevel(),
                          chunksizes=(1, min(ny, 512), min(nx, 512)))
    et.long_name = "ECHO-ET hourly evapotranspiration (ECOSTRESS-based, random-forest)"
    et.standard_name = ET_STANDARD_NAME
    et.units = ET_UNITS
    et.valid_range = ET_VALID_RANGE
    et.coordinates = "lat lon"; et.grid_mapping = "crs"
    et.cell_methods = "time: point (hour beginning, local solar time)"
    d.Conventions = "CF-1.8"
    d.title  = "ECHO-ET hourly evapotranspiration - ROI subset (CF-1.8)"
    d.source = ("ECHO-ET hourly ET, Local_solar_time version, UNH Global Ecology Group; "
                "random forest trained on ERA5-hourly reanalysis, MODIS vegetation index "
                "and ECOSTRESS 70-m ET; day/night bias-corrected with 194 eddy-covariance sites")
    d.institution = "University of New Hampshire, Global Ecology Group"
    d.references = BASE_URL + " ; product page " + PRODUCT_PAGE + " ; " + CITATION
    d.comment = ("Raw fills -32767 (water/nodata/values>1000) and -32768 (snow/ice) mapped to "
                 "_FillValue; negatives clipped to 0 per producer policy; values stored raw, "
                 "scale factor N/A. Gridded validation vs ECOSTRESS ET: r2=0.95, RMSE=29 W m-2, "
                 "PBIAS=0.08%. Fair use: inform J. Xiao (j.xiao@unh.edu) / J. Bu (jingyi.bu@unh.edu) "
                 "before any manuscript; co-authorship may apply.")
    d.history = (datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%MZ")
                 + " created by Colab pipeline; ROI-clipped, outside-ROI = _FillValue")
    d.grid_fingerprint = _grid_fingerprint()
    d.close()

def open_or_create_nc():
    global ds
    if ds is not None: return
    if os.path.exists(NC_LOCAL):
        ds = nc4.Dataset(NC_LOCAL, "a")
        if len(ds.dimensions["y"]) != GRID["ny"] or len(ds.dimensions["x"]) != GRID["nx"]:
            ds.close(); ds = None
            raise RuntimeError("Resumed .nc grid does not match new raster grid.")
        old_fp = getattr(ds, "grid_fingerprint", None)
        if old_fp and old_fp != _grid_fingerprint():
            ds.close(); ds = None
            raise RuntimeError("Resumed .nc fingerprint mismatch (transform/CRS changed).")
        _tu = getattr(ds.variables["time"], "units", "")
        if _tu != TIME_UNITS:
            ds.close(); ds = None
            raise RuntimeError("Resumed .nc time units %r != expected %r - refusing to mix." % (_tu, TIME_UNITS))
        if len(ds.variables["time"]):
            _last = float(ds.variables["time"][-1])
            _lo = float(nc4.date2num(datetime(2018, 1, 1), TIME_UNITS, TIME_CALENDAR))
            _hi = float(nc4.date2num(datetime(2030, 1, 1), TIME_UNITS, TIME_CALENDAR))
            if not (_lo <= _last <= _hi):
                ds.close(); ds = None
                raise RuntimeError("Resumed .nc last time value %r outside sane range - unit-mixed file?" % _last)
    else:
        create_nc(NC_LOCAL); ds = nc4.Dataset(NC_LOCAL, "a")

def close_nc():
    global ds
    if ds is not None: ds.sync(); ds.close(); ds = None

# ---------- resume log (buffered; identical on-disk format) + Drive helpers ----------
PROC_LOG_FLUSH_EVERY = 50
_proc_buffer = []

def queue_processed(url):
    _proc_buffer.append(url)
    if len(_proc_buffer) >= PROC_LOG_FLUSH_EVERY:
        flush_processed()

def flush_processed():
    if not _proc_buffer: return
    with open(PROCESSED_TXT, "a") as pf:
        pf.write("\n".join(_proc_buffer) + "\n")
        pf.flush(); os.fsync(pf.fileno())
    del _proc_buffer[:]

def load_processed(path):
    done = set()
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line: done.add(line)
    return done

def count_lines(p):
    return sum(1 for _ in open(p)) if os.path.exists(p) else 0

def get_resume_datetimes(path):
    keys, max_dt, bad = set(), None, []
    if not os.path.exists(path):
        return keys, max_dt
    try:
        with nc4.Dataset(path, "r") as dsr:
            if len(dsr.variables["time"]) == 0:
                return keys, max_dt
            units = dsr.variables["time"].units
            cal = getattr(dsr.variables["time"], "calendar", TIME_CALENDAR)
            for v in dsr.variables["time"][:]:
                try:
                    dt = nc4.num2date(v, units, cal)
                    dt = datetime(dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second)
                except Exception:
                    bad.append(float(v))  # torn write from a killed run: skip, do not crash
                    continue
                keys.add(timestep_key(dt))
                if max_dt is None or dt > max_dt:
                    max_dt = dt
        if bad:
            log.warning("Resume %s: skipped %d unreadable time value(s) (e.g. %s) - "
                        "run the repair snippet before trusting this file.",
                        os.path.basename(path), len(bad), bad[:3])
        else:
            log.info("Resume %s: %d timesteps (max %s)", os.path.basename(path), len(keys), max_dt)
    except Exception as e:
        log.warning("Cannot read resume point from %s: %s", path, e)
    return keys, max_dt
def get_md5(filepath, chunk_bytes=10 * 1024 * 1024):
    import hashlib
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk: break
            h.update(chunk)
    return h.hexdigest()

import threading as _th

_DRIVE_LOCK = _th.Lock()  # serialize Drive copies: concurrent hashing + FUSE writes collide


def robust_drive_copy(src, dst, max_retries=3):
    """Copy to Drive, tolerant of FUSE settle lag (stale reads, transient ENOENT).

    - Verifies src is visible on EVERY attempt (not just the first).
    - After writing, waits for the mount to settle, then re-reads dst up to
      4 times before calling it a mismatch (a single immediate re-read of a
      just-closed FUSE file routinely returns stale bytes).
    - Copies are serialized: parallel MD5 hashing while another thread writes
      through the same mount is what produces phantom mismatches.
    - Never touches src on failure; only the possibly-partial dst is removed.
    """
    with _DRIVE_LOCK:
        for attempt in range(1, max_retries + 1):
            try:
                if not os.path.exists(src):
                    raise IOError("source not visible (transient?): %s" % src)
                src_md5 = get_md5(src)
                os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
                with open(src, "rb") as fs, open(dst, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                time.sleep(10)  # let the FUSE mount settle before verifying
                if os.path.getsize(dst) != os.path.getsize(src):
                    raise IOError("size mismatch (retrying after settle)")
                dst_md5, last = None, None
                for _ in range(4):
                    try:
                        dst_md5 = get_md5(dst)
                        break
                    except OSError as e:
                        last = e
                        time.sleep(10)
                if dst_md5 is None:
                    raise IOError("dst unreadable after settle: %s" % last)
                if dst_md5 != src_md5:
                    raise IOError("MD5 mismatch after settle")
                log.info("Drive copy verified (%.1f MiB): %s",
                         os.path.getsize(dst) / 1024 ** 2, os.path.basename(dst))
                return True
            except Exception as e:
                log.warning("Drive copy attempt %d failed: %s", attempt, e)
                try:
                    if os.path.exists(dst):
                        os.remove(dst)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
                else:
                    raise RuntimeError("Drive copy failed: {}".format(e))
# ---------- dask-threaded filtering: many files open + processed at once ----------
@delayed(pure=True)
def filter_one(entry):
    # Pure per-file task (no shared state, deterministic). Never raises.
    # Returns (status, entry, payload): ok->2D array; mismatch/empty/flat->None; error->reason.
    import numpy as _np
    import rasterio as _rio
    local = entry["local"]
    try:
        with _rio.open(local) as src:
            if (src.width, src.height) != (GRID["ref_w"], GRID["ref_h"]) or src.crs != GRID["ref_crs"]:
                return ("mismatch", entry, "grid %dx%d %s" % (src.width, src.height, src.crs))
            raw = src.read(1, window=GRID["win"]).astype(_np.float32)
    except Exception as exc:
        return ("error", entry, "read failed: %s" % exc)
    out = _np.full(GRID["ny"] * GRID["nx"], FILL, dtype=_np.float32)
    v = raw.ravel()[GRID["idx"]]
    keep = _np.isfinite(v) & (v != WATER_FILL) & (v != SNOW_FILL) & (v <= MAX_VALID_VALUE)
    if not keep.any():
        return ("empty", entry, None)
    vv = v[keep]
    vv[vv < 0] = 0.0
    if vv.size >= 200:
        _vals, _counts = _np.unique(vv, return_counts=True)
        if _counts.max() / vv.size > 0.995:
            return ("flat", entry, None)
    out[GRID["idx"][keep]] = vv
    return ("ok", entry, out.reshape(GRID["ny"], GRID["nx"]))

def run_filter_batch(entries):
    # Filter files with dask threads in memory-bounded groups. Ordered (dt, url) output.
    results = []
    with tqdm(total=len(entries), desc="Dask filter", unit="file") as bar:
        for s in range(0, len(entries), DASK_BATCH):
            part = entries[s:s + DASK_BATCH]
            got = dask.compute(*[filter_one(e) for e in part])  # tuple, one result per task, in order
            results.extend(got)
            bar.update(len(part))
    results.sort(key=lambda r: (r[1]["dt"], r[1]["url"]))
    return results



In [ ]:
# ---------- parallel downloader (pipes stay hot; main thread owns resume math) ----------
N_DOWNLOAD_WORKERS = 8
PARALLEL_FILES = 64

def _fetch_one(x):
    dest = os.path.join(TMP_DIR, os.path.relpath(x["url"], BASE_URL))
    try: os.makedirs(os.path.dirname(dest), exist_ok=True)
    except OSError: pass
    if os.path.exists(dest) and not dest.endswith(".part"):
        ok0, _ = verify_tif(dest, x["size"])
        if ok0: return ("kept", x, dest)
        try: os.remove(dest)
        except OSError: pass
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if os.path.exists(dest): os.remove(dest)
            if attempt > 1:
                try:
                    if os.path.exists(dest + ".part"): os.remove(dest + ".part")
                except OSError: pass
            download(x["url"], dest, x["size"], quiet=True, session=get_session())
            ok, why = verify_tif(dest, x["size"])
            if ok: return ("ok", x, dest)
            log_corrupt({"url": x["url"], "stage": "download", "attempt": attempt, "reason": why})
        except Exception as exc:
            log_corrupt({"url": x["url"], "stage": "download", "attempt": attempt, "reason": str(exc)})
            time.sleep(min(2 ** attempt, 30))
    return ("failed", x, dest)

print("downloader ready")


In [ ]:
# ---------- hourly backup + main pipeline ----------
def backup_if_due(force=False):
    global next_backup, nc_dirty
    if not force and (time.time() - next_backup) < BACKUP_INTERVAL: return
    next_backup = time.time() + BACKUP_INTERVAL
    if not os.path.exists(NC_LOCAL): return
    if not (nc_dirty or force): return
    was_open = ds is not None
    flush_processed()
    if was_open: close_nc()
    log.info("Hourly backup -> Drive ...")
    robust_drive_copy(NC_LOCAL, NC_DRIVE_LATEST)
    with nc4.Dataset(NC_LOCAL, "r") as _r:
        _n = len(_r.variables["time"])
    _tmp_state = STATE_JSON + ".tmp"
    with open(_tmp_state, "w") as _f:
        json.dump({"timesteps": _n, "log_lines": count_lines(PROCESSED_TXT),
                   "time": datetime.now(timezone.utc).isoformat()}, _f)
    os.replace(_tmp_state, STATE_JSON)
    for _name, _local in ((URLS_TXT, os.path.join(LOCAL_NC_DIR, URLS_TXT)),
                          (TIF_TXT, os.path.join(LOCAL_NC_DIR, TIF_TXT)),
                          (PROCESSED_TXT, PROCESSED_TXT)):
        try:
            if os.path.exists(_local):
                robust_drive_copy(_local, os.path.join(DRIVE_OUT, _name))
        except Exception as _e:
            log.warning("Backup of %s failed: %s", _name, _e)
    if was_open and GRID is not None:
        try: open_or_create_nc()
        except Exception as _e: log.warning("Reopen after backup failed: %s", _e)
    nc_dirty = False

def pause_until_space():
    while disk_free() < MIN_FREE_BYTES:
        log.warning("Scratch below floor - pausing %d min.", SPACE_CHECK_INTERVAL // 60)
        time.sleep(SPACE_CHECK_INTERVAL)

def run_pipeline():
    global nc_dirty
    log.info("=" * 64)
    log.info("  ECHO-ET pipeline TEST_MODE=%s", TEST_MODE)
    for _d in (TMP_DIR, LOCAL_NC_DIR, DRIVE_OUT):
        os.makedirs(_d, exist_ok=True)
    _pairs = [(URLS_TXT, os.path.join(DRIVE_OUT, URLS_TXT)),
              (TIF_TXT, os.path.join(DRIVE_OUT, TIF_TXT)),
              (os.path.join(LOCAL_NC_DIR, LIST_CACHE), os.path.join(DRIVE_OUT, LIST_CACHE)),
              (os.path.join(LOCAL_NC_DIR, CORRUPT_LOG), os.path.join(DRIVE_OUT, CORRUPT_LOG)),
              (os.path.join(LOCAL_NC_DIR, FAILED_LOG), os.path.join(DRIVE_OUT, FAILED_LOG)),
              (os.path.join(LOCAL_NC_DIR, UNPARSED_LOG), os.path.join(DRIVE_OUT, UNPARSED_LOG)),
              (os.path.join(LOCAL_NC_DIR, SKIPPED_LOG), os.path.join(DRIVE_OUT, SKIPPED_LOG)),
              (MASK_NPZ, MASK_NPZ),
              (PROCESSED_TXT, PROCESSED_TXT)]
    for _ll, _remote in _pairs:
        if not os.path.exists(_ll) and os.path.exists(_remote):
            log.info("Restoring %s from Drive for resume...", os.path.basename(_ll))
            shutil.copy2(_remote, _ll)

    if os.path.exists(_local(LIST_CACHE)) and os.path.getsize(_local(LIST_CACHE)) > 0:
        all_files = json.load(open(_local(LIST_CACHE)))
        log.info("Loaded cached listing: %d files", len(all_files))
    else:
        log.info("Crawling server (one-time, minutes)...")
        all_files = crawl(BASE_URL)
        if not all_files:
            raise RuntimeError("Crawl found 0 files - server may be down; retry later.")
        fill_sizes(all_files)
        json.dump(all_files, open(_local(LIST_CACHE), "w"))
    if _write_list_if_changed(_local(URLS_TXT), [x["url"] for x in all_files]):
        log.info("urls.txt written (%d urls)", len(all_files))
    tif_files = [x for x in all_files if x["name"].lower().endswith(".tif")]
    tif_files.sort(key=lambda x: x["name"])
    if _write_list_if_changed(_local(TIF_TXT), [x["url"] for x in tif_files]):
        log.info("tif_urls.txt written (%d urls)", len(tif_files))
    log.info("%d urls, %d .tif (%s)", len(all_files), len(tif_files),
             human(sum(x["size"] or 0 for x in tif_files)))
    for _name in (URLS_TXT, TIF_TXT, LIST_CACHE):
        try: robust_drive_copy(_local(_name), os.path.join(DRIVE_OUT, _name))
        except Exception as _e: log.warning("%s Drive backup failed: %s", _name, _e)

    if TEST_MODE:
        tif_files = tif_files[:TEST_LIMIT]
        log.info("TEST_MODE: first %d files only", len(tif_files))
    processed = load_processed(PROCESSED_TXT)
    skipped = set()
    if os.path.exists(SKIPPED_LOG):
        for _l in open(SKIPPED_LOG):
            try: skipped.add(json.loads(_l)["url"])
            except Exception: pass
    unparsed_seen = set()
    if os.path.exists(UNPARSED_LOG):
        unparsed_seen = {l.strip() for l in open(UNPARSED_LOG) if l.strip()}
    todo, n_unparsed = [], 0
    for x in tif_files:
        dt = parse_dt(x["name"])
        if dt is None:
            if x["url"] not in unparsed_seen:
                with open(UNPARSED_LOG, "a") as _f: _f.write(x["url"] + "\n")
            n_unparsed += 1; continue
        x["dt"] = dt
        if x["url"] in processed or x["url"] in skipped: continue
        todo.append(x)
    todo.sort(key=lambda x: (x["dt"], x["url"]))
    log.info("%d files queued (%d unparseable)", len(todo), n_unparsed)

    done_keys, _ = get_resume_datetimes(NC_LOCAL)
    total_steps, pos = 0, 0
    try:
        while pos < len(todo):
            chunk = todo[pos:pos + BATCH_SIZE]; pos += BATCH_SIZE
            log.info("CHUNK: %d files", len(chunk))
            staged, pending = [], list(chunk)
            if disk_free() < MIN_FREE_BYTES: pause_until_space()
            last_space = time.time()
            pool = ThreadPoolExecutor(max_workers=N_DOWNLOAD_WORKERS)
            dl_bar = tqdm(total=len(chunk), desc="Download", unit="file")
            try:
                while pending:
                    if time.time() - last_space >= SPACE_CHECK_INTERVAL:
                        last_space = time.time()
                        if disk_free() < MIN_FREE_BYTES:
                            log.warning("Space floor hit - processing staged files first.")
                            break
                    sub = pending[:PARALLEL_FILES]; pending = pending[PARALLEL_FILES:]
                    futs = {pool.submit(_fetch_one, x): x for x in sub}
                    for fut in as_completed(futs):
                        x = futs[fut]
                        try: status, x, dest = fut.result()
                        except Exception as exc:
                            log.warning("worker error, will retry next run: %s: %s", x["url"], exc)
                            status, dest = "failed", None
                        if status in ("ok", "kept"):
                            staged.append({"url": x["url"], "local": dest,
                                           "size": x["size"], "dt": x["dt"]})
                            note_outcome(True, "download")
                        else:
                            try:
                                if dest and os.path.exists(dest + ".part"): os.remove(dest + ".part")
                            except OSError: pass
                            log_failed(x["url"])
                            note_outcome(False, "download")
                        dl_bar.update(1)
            finally:
                dl_bar.close(); pool.shutdown(wait=True)
            if pending:
                log.warning("%d files not downloaded this cycle - resume next run.", len(pending))
            staged.sort(key=lambda e: (e["dt"], e["url"]))
            if staged and not ensure_grid(staged):
                log.warning("No staged files for grid init - skipping cycle.")
            elif staged:
                open_or_create_nc()
                n_written = 0
                for status, entry, payload in run_filter_batch(staged):
                    url, dt, local = entry["url"], entry["dt"], entry["local"]
                    if status == "ok":
                        key = timestep_key(dt)
                        if key in done_keys:
                            os.remove(local); queue_processed(url)
                            note_outcome(True, "filter/append"); continue
                        try:
                            t_idx = len(ds.variables["time"])
                            ds.variables["time"][t_idx] = nc4.date2num(dt, TIME_UNITS, TIME_CALENDAR)
                            ds.variables["ET"][t_idx, :, :] = payload
                            if n_written % SYNC_EVERY == 0: ds.sync()
                            done_keys.add(key)
                            os.remove(local); queue_processed(url)
                            n_written += 1; total_steps += 1
                            note_outcome(True, "filter/append")
                        except Exception as exc:
                            log_failed(url)
                            note_outcome(False, "filter/append")
                    elif status in ("mismatch", "empty", "flat"):
                        log_skipped(url, status)
                        try:
                            if os.path.exists(local): os.remove(local)
                        except OSError: pass
                    else:
                        log_failed(url)
                        try:
                            if os.path.exists(local): os.remove(local)
                        except OSError: pass
                        note_outcome(False, "filter")
                flush_processed()
                log.info("Cycle: %d appended (total %d).", n_written, total_steps)
            for _f in glob.glob(os.path.join(TMP_DIR, "*.tif")):
                if os.path.basename(_f) in load_processed(PROCESSED_TXT):
                    try: os.remove(_f)
                    except OSError: pass
            backup_if_due()
            gc.collect()
    except CorruptStall as _e:
        print("\n" + "=" * 64)
        print("  WARNING: CORRUPTION STALL - STOPPING (no data lost)")
        print("  %s" % _e)
        print("  Partial outputs are safe on scratch + Drive. Just re-run to resume.")
        print("=" * 64 + "\n")
        return {"status": "corrupt-stall", "steps": total_steps}
    finally:
        try: flush_processed()
        except Exception: pass
        try: close_nc()
        except Exception: pass
        gc.collect()
    backup_if_due(force=True)
    try:
        with nc4.Dataset(NC_LOCAL, "r") as _r:
            _total = len(_r.variables["time"])
    except Exception: _total = total_steps
    try: robust_drive_copy(NC_LOCAL, NC_DRIVE_FINAL)
    except Exception as _e: log.warning("Final snapshot failed: %s", _e)
    log.info("PIPELINE COMPLETE - %d new steps, %d total | TEST_MODE=%s",
             total_steps, _total, TEST_MODE)
    return {"status": "done", "steps": total_steps, "total": _total}

run_pipeline()


## After the run — verify
- `!ncdump -h` on the Drive `.nc`: `Conventions = "CF-1.8"`, `ET:units = "W m-2"`,
  time `long_name` says local solar time, `grid_fingerprint` present.
- Reuse `local data processing/validate_echoet_nc.py`:
  `python validate_echoet_nc.py "<path>.nc"` (checks continuity, ranges, mask).
- Hourly continuity: 1000 files ≈ 41.7 days; any gap > 1 h is a server-side hole,
  listed by the validator.
- `failed_files.txt` entries that repeat across runs need inspection (poison source).
- Knobs if too slow: raise `N_DOWNLOAD_WORKERS` (8 → 12) only if the server
  tolerates it; raise `DASK_WORKERS`/`DASK_BATCH` for filtering; never raise both
  at once (bandwidth vs CPU contention).
- Remember the fair-use terms in Cell 1 before publishing with this data.